#### Objective: Relation Extraction
Goal: Our goal is to train a model to automatically identify relationships between entities (like "person" and "organization") within sentences. For example, in a sentence like "Steve Jobs was the CEO of Apple," we want the model to recognize that "Steve Jobs" and "Apple" have a "CEO of" relationship.
#### Step-by-Step Process
At a high level, here’s how we approach the task:

Data Preparation:

Tokenizing Sentences: We start by breaking down each sentence into tokens (individual words or subwords), so the model can understand each part of the sentence.
Labeling Relationships: Each sentence contains relationships we want the model to learn. These relationships are labeled and encoded into numerical values so the model can process them.
Model Architecture:

BERT for Sentence Understanding: We use BERT, a state-of-the-art language model, to understand the context of each sentence. BERT outputs representations for each part of the sentence, which captures the meaning of the words and their relationships within the sentence.
Classifier Layer for Relations: On top of BERT, we add a simple classification layer to determine the specific relationship type between entities (like "CEO of" or "founded by").
Training the Model:

Passing Data Through the Model: Each tokenized sentence, along with its relationships, is passed through the model. BERT processes the sentence, and the classifier layer tries to predict the correct relationship.
Handling Variable Relationships: Since sentences can have multiple relationships (or entities) of varying lengths, we added special handling with padding and masking. This ensures that the model only learns from valid relationships and ignores irrelevant padding.
Loss Calculation and Optimization:

Masked Loss: We calculate the error between the model’s prediction and the true relationship labels for each sentence. We ignore padding values in the calculation, so the model only focuses on real relationships.
Updating the Model: Based on this error, we adjust the model’s weights to improve its predictions. This process is repeated over many batches and epochs, so the model gradually learns to understand and predict relationships accurately.
#### Expected Outcome
Trained Model: At the end of training, we’ll have a model that can take any new sentence and identify relationships between entities within it. For example, it could identify that “Microsoft” and “Bill Gates” have a “founded by” relationship or that “Elon Musk” is the “CEO of” “Tesla.”
Summary
In summary, we’re building a relation extraction model that learns from labeled examples to identify entity relationships within sentences. The model leverages BERT to understand sentence context and a classifier layer to predict relationships. Through training, it learns to generalize these relationships, enabling it to make accurate predictions on new, unseen sentences.

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertModel, AdamW
from sklearn.preprocessing import LabelEncoder



###  Setting parameter on top to enable 'run all'

In [135]:
# Define training parameters
use_all_data = False
input_size = 200
use_num_rel = 3
epochs = 20
learning_rate = 1e-4


#### Load data

In [136]:
training_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_train_data.json'
test_data_path = '/Users/monilshah/Documents/02_NWU/10_MSDS_453_NLP/98_project_work/02_wip_data/relations_test_data.json'

### Get the test data

In [137]:
# Define the selected relations to filter
#selected_rel = ['industry', 'product_or_material_produced', 'owned_by']

selected_rel = [ 'product_or_material_produced', 'industry']

## Reason why I am getting test data now is because for encoder I want all entities and all relations

# Load the test data
with open(test_data_path, 'r') as file:
    test_data = json.load(file)
    
test_data = test_data[0]


# Filter the data
filtered_test_data = []
for entry in test_data:  
    # Check if the entry has a "triples" field and filter based on selected_rel
    if isinstance(entry, dict) and "triples" in entry:
        matching_triples = [
            triple for triple in entry["triples"]  # Loop through the triples list
            if isinstance(triple, dict) and triple.get("relation") in selected_rel
        ]
        # If there are matching triples, add the entry to filtered_data with relevant triples
        if matching_triples:
            filtered_entry = {
                "news_line": entry["news_line"],
                "triples": matching_triples
            }
            filtered_test_data.append(filtered_entry)

test_data = filtered_test_data


# Prepare the test data
test_sentences = []
test_relation_labels = []


# Extract sentences and labels from the test data
for entry in test_data:
    news_line = entry['news_line']
    test_sentences.append(news_line)

    # Extract relations and split them into separate entries
    relations = [triple["relation"] for triple in entry["triples"]]
    test_relation_labels.append(relations)  # Flatten the list


### Get the training data

In [138]:
import json

# Load the JSON file
with open(training_data_path, 'r') as file:
    all_data = json.load(file)

# Example structure of each data entry
# {
#     "news_line": "Sentence text here.",
#     "triples": [
#         {"subject": "Entity1", "object": "Entity2", "relation": "RelationType"}
#     ]
# }



In [139]:

# Load the JSON file
with open(training_data_path, 'r') as file:
    all_data = json.load(file)

all_data = all_data[0]

if not use_all_data:
    # Filter the data
    filtered_data = []
    for entry in all_data:  
        # Check if the entry has a "triples" field and filter based on selected_rel
        if isinstance(entry, dict) and "triples" in entry:
            matching_triples = [
                triple for triple in entry["triples"]  # Loop through the triples list
                if isinstance(triple, dict) and triple.get("relation") in selected_rel
            ]
            # If there are matching triples, add the entry to filtered_data with relevant triples
            if matching_triples:
                filtered_entry = {
                    "news_line": entry["news_line"],
                    "triples": matching_triples
                }
                filtered_data.append(filtered_entry)

    # Print the number of filtered entries
    print(f"Number of entries with selected relations: {len(filtered_data)}")

    # Save the filtered data to a new JSON file (optional)
    filtered_data_path = "filtered_training_data.json"
    with open(filtered_data_path, 'w') as file:
        json.dump(filtered_data, file, indent=4)

    print(f"Filtered data saved to {filtered_data_path}")
    all_data = filtered_data


Number of entries with selected relations: 2272
Filtered data saved to filtered_training_data.json


### Prepare Data

Tokenization: We tokenized each sentence using a BERT tokenizer to get input_ids and attention_mask for each sentence.

Label Encoding: We encoded each relation type as an integer using LabelEncoder and padded the relation labels to ensure consistent tensor sizes across sentences.

Masking: We used -1 as a padding value for relation labels, creating a mask to identify valid (non-padded) relation labels in each batch.

In [140]:


# Assuming `data` is already loaded and contains 'news_line' and 'triples'
data = all_data  # Since the whole document is loaded as one JSON object

# Prepare the dataset for tokenization and labeling
sentences = []
relation_labels = []

# Iterate through each entry in the data
for entry in data:
    news_line = entry['news_line']
    sentences.append(news_line)

    # Initialize relation labels
    relations = [triple["relation"] for triple in entry["triples"]]
    relation_labels.append(relations)  # List of relations for the current entry





In [141]:
from collections import Counter

# Flatten the list of lists
flattened_labels = [relation for sublist in relation_labels for relation in sublist]

# Count occurrences
relation_counts = Counter(flattened_labels)

# Display counts
print("Relation Counts:")
for relation, count in relation_counts.items():
    print(f"{relation}: {count}")


Relation Counts:
product_or_material_produced: 1373
industry: 1167


In [142]:
# Initialize label encoder for relations
relation_label_encoder = LabelEncoder()


#all_relations = [relation for sublist in relation_labels for relation in sublist + relations for sublist in test_relation_labels for relation in sublist ]

# Combine and flatten relation labels
all_relations = [
    relation
    for sublist in (relation_labels + test_relation_labels)
    for relation in sublist
]


relation_label_encoder.fit(all_relations)

# Encode relations
encoded_relations = [relation_label_encoder.transform(rel) for rel in relation_labels]



In [143]:

# Ensure all entries in encoded_relations, encoded_entity1, and encoded_entity2 are of the same length
max_relations = 1  # Set your desired maximum number of relations
padded_relations = []


for relations in (encoded_relations):
    # Pad or truncate relations
    if len(relations) < max_relations:
        padded_relations.append(relations.tolist() + [-1] * (max_relations - len(relations)))  # Pad with -1
    else:
        padded_relations.append(relations[:max_relations].tolist())  # Truncate if too long

    

## Create batch

In [144]:
# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


# Tokenize the texts
encoding = tokenizer(
    sentences,
    padding="max_length",  # Ensure all sequences are of the same length
    truncation=True,       # Truncate sequences longer than `max_length`
    max_length=32,        # Set the maximum token sequence length
    return_tensors="pt"    # Return tensors (PyTorch format)
)

input_ids_tensor = encoding["input_ids"]  # Input IDs for the model
attention_masks_tensor = encoding["attention_mask"]  # Attention masks for the model


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert padded relations and entities to tensors
relation_labels_tensor = torch.tensor(padded_relations, dtype=torch.long)


# Print shapes for verification
print("Padded Relation Labels shape:", relation_labels_tensor.shape)

# Create the TensorDataset with relations and entities
train_dataset = TensorDataset(input_ids_tensor, attention_masks_tensor, relation_labels_tensor)

# Create DataLoader for the training set
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# Print sample from the DataLoader to verify
for batch in train_loader:
    print(batch)
    break  # Remove this break to iterate through the entire dataset


Padded Relation Labels shape: torch.Size([2272, 1])
[tensor([[  101,  1999,  1996,  2048,  1011, 12819,  6903,  1010, 14278,  9055,
          5068,  1037,  3943,  3867,  3623,  1999,  2037,  2251,  2325,  4341,
          2012,  4029,  1010,  5511,  2487,  3197,  2013,  2676,  1010, 12963,
          3197,   102]]), tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]]), tensor([[1]])]


## Define model


### **1. The Goal of the Model**
This model, `TripletExtractionModel`, uses a BERT-based architecture to extract relationships between entities in a sentence. It predicts relations and can be extended to predict entities too (though not fully implemented in the given code).

---

### **2. Breaking It Down**

#### **Model Initialization (`__init__`)**
- **`self.bert`**: 
  - Loads a pre-trained BERT model (`bert-base-uncased`) that provides contextual embeddings for input text.
  - This model processes the text and provides a deep understanding of its meaning.
  
- **Relation Classifier (`self.relation_classifier`)**:
  - A simple linear layer (fully connected layer) that takes the output from BERT and maps it to `num_relations` possible relation types. 
  - Example: If there are 30 relation types, this layer outputs 30 scores for each input (one for each type).

- **`max_relations`**:
  - Represents the maximum number of relations the model expects to extract per input.

---

#### **Forward Method (`forward`)**
This defines the steps the model takes when processing an input.

1. **Input to BERT**:
   - `input_ids` and `attention_mask` are fed into the BERT model.
   - **`input_ids`**: Tokenized words from the text (numbers that represent words).
   - **`attention_mask`**: Identifies which tokens are actual words and which are padding (ignored by BERT).
   - BERT produces `outputs.last_hidden_state`, which is a matrix containing embeddings (dense representations) for each token in the input.

2. **Extract CLS Token Output**:
   - The **CLS token** (first token in BERT input) is special. It summarizes the entire input's meaning for tasks like classification.
   - `outputs.last_hidden_state[:, 0, :]` extracts the CLS token's embedding for all input sequences in the batch.

3. **Expand CLS Output**:
   - The CLS output is reshaped to allow the model to predict multiple relations for each input if needed.
   - `unsqueeze` and `expand` adjust its shape for compatibility with `max_relations`.

4. **Predict Relations**:
   - The expanded CLS embeddings are fed through the relation classifier (`self.relation_classifier`), which outputs logits (scores) for each relation type.

5. **Flatten the Output**:
   - The output is reshaped to simplify loss calculation during training. 
   - After reshaping, the shape is `(batch_size * max_relations, num_relations)` where:
     - **`batch_size * max_relations`**: Total number of relation predictions.
     - **`num_relations`**: Number of possible relation classes.

---

### **Output**
The method returns `relation_logits`, which contains the raw scores for each relation type. These scores can later be converted into probabilities using softmax, and the most probable relation can be chosen.

---

### **Simplified Workflow**
1. Input a sentence → Process with BERT → Extract CLS output (summary of sentence meaning).
2. Expand CLS output → Use it to predict relation scores.
3. Flatten predictions for easy loss calculation during training.

This model's job is essentially to classify relationships in a text, leveraging the powerful contextual understanding of BERT.

In [146]:
import torch
import torch.nn as nn
from transformers import BertModel

class TripletExtractionModel(nn.Module):
    def __init__(self, model_name="bert-base-uncased", max_relations=1, num_relations=30):
        super(TripletExtractionModel, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.max_relations = max_relations
        
        # Classifiers for relations, entity1, and entity2
        self.relation_classifier = nn.Linear(self.bert.config.hidden_size, num_relations)
        
    def forward(self, input_ids, attention_mask):
        # Get outputs from BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Get the CLS token output for each input sequence
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token output
        
        # Expand CLS output to have a shape (batch_size, max_relations, hidden_size)
        cls_output_expanded = cls_output.unsqueeze(1).expand(-1, self.max_relations, -1)
        
        # Classify relations, entity1, and entity2 using the expanded CLS output
        relation_logits = self.relation_classifier(cls_output_expanded)
        
        # Reshape the output to (batch_size * max_relations, num_classes) for loss calculation
        relation_logits = relation_logits.view(-1, relation_logits.size(-1))  # Flatten for relation loss
        
        return relation_logits


### Train Model



### **1. Model and Optimizer Setup**
- **Initialize the model**:
  - `TripletExtractionModel` is initialized with `num_relations`, which specifies the number of relation types the model will predict. This value is determined using `relation_label_encoder.classes_` (a label encoder that maps relation names to numerical labels).

- **Optimizer**:
  - `AdamW` is used as the optimizer. It's specifically designed for training models like BERT, as it handles weight decay (regularization) efficiently.

- **Device Selection**:
  - Checks if a GPU (CUDA) is available. If yes, the model will use the GPU for faster training. Otherwise, it defaults to the CPU.

---

### **2. Training Loop**
This loop runs for a specified number of `epochs` (full passes through the training data). Here’s what happens inside the loop:

#### **a. Prepare for Training**
- **Switch to training mode**:
  - `model.train()` activates dropout layers and other training-specific behavior.
  
- **Initialize `total_loss`**:
  - Tracks the cumulative loss for the epoch.

---

#### **b. Process Each Batch**
For each batch of data from `train_loader`:

1. **Extract Inputs and Labels**:
   - `input_ids`: Encoded tokens of input sentences.
   - `attention_mask`: Indicates which tokens are valid (not padding).
   - `relation_labels`: Ground truth labels for relations.

2. **Move Data to Device**:
   - Moves the input data (`input_ids`, `attention_mask`, `relation_labels`) to the same device (GPU/CPU) as the model.

3. **Forward Pass**:
   - The model processes the inputs, producing `relation_logits` (raw scores for relation predictions).

4. **Reshape Outputs and Labels**:
   - Reshapes the model's output (`relation_logits`) and ground truth labels (`relation_labels`) so they can be compared. This ensures compatibility with the `CrossEntropyLoss` function.

5. **Define and Calculate Loss**:
   - **Loss function**: `CrossEntropyLoss` is used for classification. 
     - `ignore_index=-1`: Skips any labels with a value of `-1`, which might represent padding.
   - Calculates the relation classification loss (`relation_loss`).

6. **Backpropagation and Optimization**:
   - **Zero out gradients**:
     - `optimizer.zero_grad()` clears the accumulated gradients from the previous batch.
   - **Compute gradients**:
     - `loss.backward()` computes the gradients of the loss with respect to the model parameters.
   - **Update model parameters**:
     - `optimizer.step()` updates the model's weights using the gradients.

7. **Track Loss**:
   - Adds the batch's loss value to `total_loss`.

---

#### **c. Log Epoch Progress**
- At the end of each epoch, calculates the average loss (`avg_loss`) for all batches.
- Prints the average loss for monitoring training progress.

---

### **3. Purpose of the Code**
This code trains the `TripletExtractionModel` to predict relations between entities in text. It:
1. Passes batches of input text through the model.
2. Compares the model's predictions to the true relation labels.
3. Updates the model to improve its predictions by minimizing the loss over multiple epochs.

---

### **Simplified Workflow**
1. Data → Batch → Model predicts relation logits.
2. Compute loss comparing predictions and ground truth.
3. Adjust model weights to reduce the loss.
4. Repeat for all batches and epochs.

By the end of training, the model should be better at predicting relations based on the training data.

In [153]:

# Define an early stopping callback
class EarlyStoppingCallback:
    def __init__(self, patience=3, min_delta=0, save_path=None):
        """
        Args:
        - patience (int): Number of epochs with no improvement after which training will stop.
        - min_delta (float): Minimum change in the monitored quantity to qualify as an improvement.
        - save_path (str): Path to save the best model. If None, no model is saved.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.best_loss = float('inf')  # Initialize to infinity
        self.counter = 0  # Counter for epochs with no improvement

    def on_epoch_end(self, epoch, logs=None):
        """
        Called at the end of each epoch.
        Args:
        - epoch (int): Current epoch number.
        - logs (dict): Dictionary containing 'avg_loss' and 'model_state'.
        """
        current_loss = logs['avg_loss']
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.counter = 0  # Reset counter on improvement
            print(f"[EarlyStopping] Loss improved to {self.best_loss:.4f}.")
            # Save the best model if save_path is specified
            if self.save_path:
                os.makedirs(self.save_path, exist_ok=True)  # Ensure the directory exists
                torch.save(logs['model_state'], f"{self.save_path}/best_model.pt")
        else:
            self.counter += 1
            print(f"[EarlyStopping] No improvement for {self.counter}/{self.patience} epochs.")

        if self.counter >= self.patience:
            print("[EarlyStopping] Stopping training as no improvement observed.")
            return True  # Signal to stop training
        return False

# Initialize the model and optimizer
num_relations = len(relation_label_encoder.classes_)  # Assuming label_encoder is defined and fitted for relations
model = TripletExtractionModel(num_relations=num_relations)
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Move model to the appropriate device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize the early stopping callback
early_stopping = EarlyStoppingCallback(patience=3, save_path="checkpoints")

# Training loop
model.train()
for epoch in range(epochs):
    total_loss = 0  # To accumulate loss over each epoch
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids, attention_mask, relation_labels = batch  # Separate entity labels

        # Move batch to the appropriate device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        relation_labels = relation_labels.to(device)
        
        # Forward pass
        relation_logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # Reshape logits and labels to match dimensions for CrossEntropyLoss
        relation_logits = relation_logits.view(-1, num_relations)  # For relation loss calculation
        relation_labels = relation_labels.view(-1)
        
        # Define loss functions with ignore_index=-1 for padding
        relation_loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
        
        # Calculate individual losses
        relation_loss = relation_loss_fn(relation_logits, relation_labels)
        
        # Total loss combines relation, entity1, and entity2 losses
        loss = relation_loss 

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()  # Accumulate loss

    # Average loss for the epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")
    
    # Early stopping check
    stop_training = early_stopping.on_epoch_end(epoch, logs={'avg_loss': avg_loss, 'model_state': model.state_dict()})
    if stop_training:
        break


/opt/anaconda3/envs/Conda_3_12_7/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/20 - Average Loss: 0.7153
[EarlyStopping] Loss improved to 0.7153.
Epoch 2/20 - Average Loss: 0.7009
[EarlyStopping] Loss improved to 0.7009.
Epoch 3/20 - Average Loss: 0.6967
[EarlyStopping] Loss improved to 0.6967.
Epoch 4/20 - Average Loss: 0.6920
[EarlyStopping] Loss improved to 0.6920.
Epoch 5/20 - Average Loss: 0.6900
[EarlyStopping] Loss improved to 0.6900.
Epoch 6/20 - Average Loss: 0.6906
[EarlyStopping] No improvement for 1/3 epochs.
Epoch 7/20 - Average Loss: 0.6916
[EarlyStopping] No improvement for 2/3 epochs.
Epoch 8/20 - Average Loss: 0.6906
[EarlyStopping] No improvement for 3/3 epochs.
[EarlyStopping] Stopping training as no improvement observed.


### Save Model

In [154]:
# After the training loop, save the model
torch.save(model.state_dict(), "trained_relation_extraction_model.pth")
print("Model saved successfully!")


Model saved successfully!


### Test Model

In [155]:
import torch
import json
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer

# Define the model and load the state dictionary
model = TripletExtractionModel(num_relations=num_relations)
model.load_state_dict(torch.load("trained_relation_extraction_model.pth"))
model.eval()  # Set the model to evaluation mode

# Move model to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


/var/folders/ws/v15jk6j56z90j1tz393rgysw0000gn/T/ipykernel_27972/1048299488.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("trained_rel

TripletExtractionModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [156]:
# Flatten nested lists for transformation
flat_relation_labels = [label for sublist in test_relation_labels for label in sublist]


# Encode with the label encoder
encoded_flat_relations = relation_label_encoder.transform(flat_relation_labels)

# Restore the original structure of nested lists after encoding
encoded_test_relations = []


index = 0
for sublist in test_relation_labels:
    encoded_test_relations.append(encoded_flat_relations[index:index + len(sublist)])
    index += len(sublist)


# Padding or truncating for a uniform shape
padded_relations = []
max_relations = 1  # Adjust as per requirement

for relations in (encoded_test_relations):
    # Pad or truncate relations
    relations = relations.tolist()
    if len(relations) < max_relations:
        padded_relations.append(relations + [-1] * (max_relations - len(relations)))
    else:
        padded_relations.append(relations[:max_relations])

    

In [157]:

# Tokenize the texts
test_encoding = tokenizer(
    test_sentences,
    padding="max_length",  # Ensure all sequences are of the same length
    truncation=True,       # Truncate sequences longer than `max_length`
    max_length=32,        # Set the maximum token sequence length
    return_tensors="pt"    # Return tensors (PyTorch format)
)

input_ids_tensor = test_encoding["input_ids"]  # Input IDs for the model
attention_masks_tensor = test_encoding["attention_mask"]  # Attention masks for the model


In [158]:
relation_labels_tensor.shape, input_ids_tensor.shape, attention_masks_tensor.shape

(torch.Size([2272, 1]), torch.Size([336, 32]), torch.Size([336, 32]))

In [159]:
# Convert test data to tensors
#input_ids_tensor = torch.randint(0, 100, (len(padded_relations), 122))  # Simulated input IDs, replace with actual data if available
#attention_masks_tensor = torch.ones(input_ids_tensor.shape, dtype=torch.long)  # Simulated attention masks, replace as needed

relation_labels_tensor = torch.tensor(padded_relations, dtype=torch.long)

# Create DataLoader for test data
test_dataset = TensorDataset(input_ids_tensor, attention_masks_tensor, relation_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Run inference on the test data
model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, relation_labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Forward pass
        relation_logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # Convert logits to predictions
        relation_preds = torch.argmax(relation_logits, dim=1)
        
        # Print or log the predictions
        print("Relation Predictions:", relation_preds)
        
        # Add any further processing to display these results or match them with original input if needed


Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predictions: tensor([1])
Relation Predi

In [160]:
# Initialize empty lists to collect predictions
all_relation_preds = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, relation_labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Forward pass
        relation_logits = model(input_ids=input_ids, attention_mask=attention_mask)

        # Convert logits to predictions
        relation_preds = torch.argmax(relation_logits, dim=1).cpu().numpy()
        
        # Append predictions to the lists
        all_relation_preds.extend(relation_preds)
        
# Transform predictions back to original labels
original_relation_preds = relation_label_encoder.inverse_transform(all_relation_preds)

# Print the results
print("Original Relation Predictions:", original_relation_preds)



Original Relation Predictions: ['product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_produced' 'product_or_material_produced'
 'product_or_material_pr